In [0]:
# Force uninstall both to clear the 'google' namespace
%pip uninstall -y google-generativeai google-genai

# Reinstall only the modern SDK
%pip install google-genai

# CRITICAL: Restart the Python REPL to clear the cached imports
dbutils.library.restartPython()

In [0]:
import google
print(f"Google package location: {google.__path__}")

try:
    from google import genai
    print("Success! genai is imported.")
except ImportError as e:
    print(f"Import failed: {e}")

In [0]:
def generate_with_fallback(prompt):
    models_to_try = ["gemini-3-pro-preview", "gemini-2.5-pro", "gemini-2.5-flash"]
    
    for model_name in models_to_try:
        try:
            print(f"Attempting with {model_name}...")
            # Your existing client.models.generate_content_stream logic here
            # ...
            return # Exit if successful
        except exceptions.ResourceExhausted:
            print(f"Quota exceeded for {model_name}. Trying fallback...")
            continue 
    print("All models exhausted. Please wait for quota reset (Midnight PT).")

In [0]:
from pyspark.sql.functions import concat_ws

# Loading your 5000 records
df = spark.read.option("header", "true").csv("/Volumes/raw-data/banking/csv/Banking_Database.csv")

# Create the text document for embeddings
text_df = df.withColumn(
    "document",
    concat_ws(" | ", "Customer ID", "First Name", "Last Name", "Age", "Account Type", "Loan Status", "Anomaly")
)

# Convert to list for embedding (Collect is fine for 5k records)
documents = [row.document for row in text_df.select("document").collect()]

In [0]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load Model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate Embeddings
embeddings = embedding_model.encode(documents)

# Create FAISS Index (L2 Distance)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

print(f"Total indexed records: {index.ntotal}")

In [0]:
import time
import numpy as np
from google import genai
from google.genai import types
from google.api_core import exceptions

# Configure the client outside the function for better performance
# Use an environment variable for security in your Databricks project
client = genai.Client(api_key="api key ")

def ask_banking_ai(question):
    try:
        # 1. Search Vector DB for context
        q_embedding = embedding_model.encode([question])
        # We only need the top 3-5 records to stay under the Token Limit (TPM)
        D, I = index.search(np.array(q_embedding).astype('float32'), k=3) 
        
        retrieved_context = "\n".join([documents[i] for i in I[0]])
        
        # 2. Setup Prompt
        prompt = f"Context: {retrieved_context}\n\nQuestion: {question}"
        
        # 3. Model Configuration (Optimized for Free Tier)
        # Using 2.5-flash because it has a generous free tier in 2026
        model_id = "gemini-2.5-flash" 
        
        # Optimized configuration for Gemini 2.5 Flash
        generate_content_config = types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(
                include_thoughts=True, 
                # For 2.5 models, we use a token budget instead of "LOW/HIGH"
                # 1024 is a good balance for data extraction logic
                thinking_budget=1024 
            ),
            temperature=0.0  # Keep at 0 for accurate banking data
        )

        # 4. Call the model
        response = client.models.generate_content(
            model=model_id,
            contents=prompt,
            config=generate_content_config
        )
        
        return response.text

    except exceptions.ResourceExhausted:
        return "Error: API Quota exceeded. Please wait 60 seconds."
    except Exception as e:
        return f"An error occurred: {str(e)}"

# --- Execution Logic for 5,000 records ---
# If you are looping through questions, add a delay to avoid 429
questions = ["Which customers require immediate risk review?"]

for q in questions:
    print(f"Query: {q}")
    print(f"Response: {ask_banking_ai(q)}")
    print("-" * 30)
    # The Free Tier allows ~10-15 requests per minute. 
    # A 6-second sleep ensures you never hit the 429 error.
    time.sleep(6)